In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('ecommerce_retail_transactions_raw.csv')

df.head()
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [3]:
df.columns = df.columns.str.strip().str.lower().str.replace({'unit_price_usd':'unit_price',
                                                             'discount_percent':'discount_pct'})

In [4]:
df['order_date'] = pd.to_datetime(df['order_date'],format='mixed',errors='coerce')

In [5]:
df['product_category'] = df['product_category'].str.strip().str.lower()
df['product_category'].unique()

<ArrowStringArray>
[         'books', 'home & kitchen',         'beauty',        'fashion',
         'sports',           'toys',    'electronics',        'grocery']
Length: 8, dtype: str

In [6]:
df['product_name'] = df['product_name'].str.strip().str.lower()
df['product_name'].unique()

<ArrowStringArray>
[     'self-help book',       'fiction novel',      'vacuum cleaner',
          'hair dryer',       'non-stick pan',   'storage organizer',
        'formal shirt',            'cookbook',        'dumbbell set',
      'kids storybook',     'sunscreen spf50',  'remote control car',
       'action figure',      'cycling helmet',              'webcam',
      'cotton t-shirt', 'building blocks set',         'cricket bat',
            'football',        'gaming mouse',    'basmati rice 5kg',
    'wireless earbuds',            'yoga mat',             'blender',
      'green tea pack',             'shampoo',   'bluetooth speaker',
        'almonds 500g',           'air fryer',            'lipstick',
          'smartwatch',          'sunglasses',      'leather wallet',
     'ceramic mug set',       'puzzle 1000pc',        'olive oil 1l',
    'face moisturizer',            'backpack',       'running shoes',
             'perfume',        'laptop stand',          'power bank',
 

In [7]:
df.loc[(df['quantity'] < 0) | (df['quantity'] == 0),'quantity'] = np.nan
df['quantity'].unique()

array([ 3.,  1.,  2.,  4.,  5., nan])

In [8]:
df['unit_price'].unique()
(df['unit_price'] < 0).any()

np.False_

In [9]:
df['discount_pct'] = df['discount_pct']/100
df['discount_pct'].unique()
(df['discount_pct'] > 1).any()

np.False_

In [10]:
df['payment_method'] = df['payment_method'].str.strip().str.lower().str.replace({'netbanking': 'net banking',
    'net banking': 'net banking',
    'debit card': 'debit card',
    'cash on delivery': 'cash on delivery',
    'cod': 'cash on delivery',
    'credit_card': 'credit card',
    'credit card': 'credit card',
    'u.p.i': 'upi',
    'upi': 'upi',
    'pay pal': 'paypal',
    'paypal': 'paypal'})
    
df['payment_method'].unique() 

<ArrowStringArray>
['net banking', 'debit card', 'cash on delivery', 'credit card', 'upi',
 'paypal']
Length: 6, dtype: str

In [11]:
df['shipping_city'] = df['shipping_city'].str.strip().str.lower()
df['shipping_city'].unique()

<ArrowStringArray>
[     'london',     'toronto',    'brisbane',  'birmingham',    'montreal',
 'los angeles',    'new york',      'munich',   'melbourne',  'manchester',
     'chicago',       'delhi',     'houston',     'hamburg',      'mumbai',
       'dubai',      'berlin',   'bengaluru',   'abu dhabi',      'sydney',
           nan,   'vancouver',   'hyderabad']
Length: 23, dtype: str

In [12]:
df['country'] = df['country'].str.strip().str.lower().replace({
    'u.k': 'uk',
    'u.k.':'uk',
    'united kingdom': 'uk',
    'u.s.a': 'usa',
    'us': 'usa',
    'united states': 'usa',
    'u.a.e': 'uae',
    'ca': 'canada',
    'de': 'germany',
    'au': 'australia',
    'in': 'india'
})

df['country'].unique()

<ArrowStringArray>
['uk', 'canada', 'australia', 'usa', 'germany', 'india', 'uae']
Length: 7, dtype: str

In [13]:
df['order_status'] = df['order_status'].str.strip().str.lower()
df['order_status'].unique()

<ArrowStringArray>
['delivered', 'shipped', 'cancelled', 'returned', 'pending']
Length: 5, dtype: str

In [14]:
df['customer_rating'].unique()

array([ 5., nan,  3.,  4.,  2.,  1.])

In [15]:
df['quantity'] = df['quantity'].astype('Int64')

In [16]:
df['quantity'] = df.groupby(['product_category','product_name'])['quantity'].transform(lambda x: x.fillna(x.median()))
df['discount_pct'] = df.groupby(['product_category','product_name'])['quantity'].transform(lambda x: x.fillna(x.median()))
df['shipping_city'] = df.groupby(['country'])['shipping_city'].transform(lambda x: x.fillna(x.mode()[0]))
df['customer_rating'] = df.groupby(['product_category','product_name','order_status'])['customer_rating'].transform(lambda x: x.fillna(x.median()))

In [17]:
df['discount_pct'] = df['discount_pct'].astype(float)
df['discount_pct'] = df['discount_pct']/100

In [22]:
df['revenue'] = df['unit_price'] * df['quantity'] * (1 - df['discount_pct'])
(df['revenue'] < 0).any()

np.False_

In [19]:
df.isna().sum()

order_id               0
customer_id            0
order_date             0
product_category       0
product_name           0
quantity               0
unit_price             0
discount_pct           0
payment_method         0
shipping_city          0
country                0
order_status           0
customer_rating     5422
revenue                0
dtype: int64

In [20]:
df.dtypes

order_id                       str
customer_id                    str
order_date          datetime64[us]
product_category               str
product_name                   str
quantity                     Int64
unit_price                 float64
discount_pct               float64
payment_method                 str
shipping_city                  str
country                        str
order_status                   str
customer_rating            float64
revenue                    Float64
dtype: object

In [21]:
df.to_csv('clean_ecom.csv',index=False)